# 3D Z^0 / Pi_h^0 with NGSolve/Netgen

This notebook generates a tetrahedral unit-cube mesh, builds the 3D Alfeld split, computes local $\psi_0(y)$ and $Z^0(y)$ for every original vertex, assembles $P_0:V_{h,A}^0\to V_h^0$, and checks $P_0E=I$.


In [ ]:
# Run this in Colab first if ngsolve is not installed:
# !pip -q install ngsolve


In [ ]:
"""
3D Z^0 / Pi_h^0 construction on tetrahedral meshes.

Colab usage:
    !pip -q install ngsolve
    %run ngsolve_z0_pi0_3d_colab.py
    out = run_demo(maxh=0.45, make_plots=True)

Mathematics:
    Pi_h^0 v = sum_i ( int_Omega Z^0(y_i) v dx ) phi_i,

where phi_i is the original tetrahedral P1 nodal basis at vertex y_i.
For one vertex y,

    Z^0(y) = eta^0(y) - div(mu_y grad psi_0(y)),
    eta^0(y) = 1/|st(y)| on st(y),

and psi_0(y) solves on the vertex star

    int_st mu_y grad psi_0(y) . grad u dx
      = u(y) - (1/|st(y)|) int_st u dx,

with zero mean on st(y).

This script builds a finite-dimensional matrix representation

    P0 : Alfeld P1 nodal values -> original P1 nodal values,
    P0(i,j) = int Z^0(y_i) Phi_j^A dx.

No essential boundary conditions are imposed. Boundary vertices are included.
"""

import numpy as np
from scipy.sparse import coo_matrix, csc_matrix, eye
from scipy.sparse.linalg import norm as sparse_norm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def make_structured_unit_cube_tets(n=2):
    """Fallback tetra mesh: split each cube in an n x n x n grid into 6 tetrahedra."""
    xs = np.linspace(0.0, 1.0, n+1)
    coords = []
    node_id = {}
    for k,z in enumerate(xs):
        for j,y in enumerate(xs):
            for i,x in enumerate(xs):
                node_id[(i,j,k)] = len(coords)
                coords.append((x,y,z))
    coords = np.array(coords, dtype=float)

    # Freudenthal splitting of a cube into 6 tetrahedra using all permutations.
    perms = [
        ((1,0,0),(1,1,0),(1,1,1)),
        ((1,0,0),(1,0,1),(1,1,1)),
        ((0,1,0),(1,1,0),(1,1,1)),
        ((0,1,0),(0,1,1),(1,1,1)),
        ((0,0,1),(1,0,1),(1,1,1)),
        ((0,0,1),(0,1,1),(1,1,1)),
    ]
    # The above alone can miss orientation consistency; orient_tets fixes it.
    tets = []
    for k in range(n):
        for j in range(n):
            for i in range(n):
                v000 = node_id[(i,j,k)]
                # Use the 6 monotone path tetrahedra from v000 to v111.
                for a,b,c in perms:
                    va = node_id[(i+a[0], j+a[1], k+a[2])]
                    vb = node_id[(i+b[0], j+b[1], k+b[2])]
                    vc = node_id[(i+c[0], j+c[1], k+c[2])]
                    tets.append([v000, va, vb, vc])
    tets = np.array(tets, dtype=int)
    return coords, orient_tets_positive(coords, tets)


def _coords_from_point_obj(pt):
    """Best-effort conversion of Netgen/NGSolve point-like objects to length-3 list."""
    obj = getattr(pt, "p", pt)
    obj = getattr(obj, "point", obj)
    try:
        return [float(obj[i]) for i in range(3)]
    except Exception:
        return [float(obj.x), float(obj.y), float(obj.z)]


def _ids_from_element_obj(el, npts):
    """Best-effort conversion of Netgen/NGSolve element object to zero-based vertex ids."""
    verts = None
    for name in ("vertices", "points"):
        if hasattr(el, name):
            verts = getattr(el, name)
            break
    if verts is None:
        verts = el

    ids = []
    for v in verts:
        if hasattr(v, "nr"):
            ids.append(int(v.nr))
        else:
            ids.append(int(v))
    ids = ids[:4]

    # Netgen point ids are often 1-based; NGSolve VertexId.nr is usually 0-based.
    if len(ids) == 4 and min(ids) >= 1 and max(ids) <= npts:
        ids = [q-1 for q in ids]
    return ids


def make_ngsolve_unit_cube_tetmesh(maxh=0.45):
    """Generate a unit cube tetra mesh with Netgen/NGSolve and extract arrays."""
    from netgen.csg import CSGeometry, OrthoBrick, Pnt
    from ngsolve import Mesh

    geo = CSGeometry()
    geo.Add(OrthoBrick(Pnt(0,0,0), Pnt(1,1,1)).bc("outer"))
    ngm = geo.GenerateMesh(maxh=maxh)
    mesh = Mesh(ngm)

    # Prefer Netgen mesh arrays because they expose volume elements directly.
    coords = None
    tets = None

    try:
        pts = list(ngm.Points())
        coords = np.array([_coords_from_point_obj(p) for p in pts], dtype=float)
        els = list(ngm.Elements3D())
        tets = np.array([_ids_from_element_obj(el, len(coords)) for el in els], dtype=int)
    except Exception:
        pass

    if coords is None or tets is None or len(tets) == 0:
        # Fallback to NGSolve mesh iterators.
        verts = list(mesh.vertices)
        coords = np.array([_coords_from_point_obj(v) for v in verts], dtype=float)
        els = list(mesh.Elements())
        tets = []
        for el in els:
            ids = _ids_from_element_obj(el, len(coords))
            if len(ids) == 4:
                tets.append(ids)
        tets = np.array(tets, dtype=int)

    tets = orient_tets_positive(coords, tets)
    return coords, tets, mesh


def make_mesh(maxh=0.45, fallback_n=2):
    """Try NGSolve/Netgen first; fall back to a small structured tetra mesh."""
    try:
        coords, tets, mesh = make_ngsolve_unit_cube_tetmesh(maxh=maxh)
        source = "NGSolve/Netgen unit cube"
        return coords, tets, mesh, source
    except Exception as err:
        print("WARNING: could not generate/extract NGSolve mesh; using structured fallback.")
        print("         NGSolve error was:", repr(err))
        coords, tets = make_structured_unit_cube_tets(n=fallback_n)
        return coords, tets, None, "structured fallback unit cube"


def tet_volume_grads(X):
    """Return positive volume and gradients of barycentric coordinates on a tet.

    X is 4 x 3. grads[i,:] = grad(lambda_i).
    """
    J = np.vstack((X[1]-X[0], X[2]-X[0], X[3]-X[0]))
    detJ = np.linalg.det(J)
    vol = detJ / 6.0
    if vol <= 0:
        raise ValueError("tet has nonpositive orientation/volume")

    M = np.vstack((X.T, np.ones(4)))
    Minv = np.linalg.inv(M)
    grads = Minv[:, 0:3]
    return vol, grads


def orient_tets_positive(coords, tets):
    tets = np.asarray(tets, dtype=int).copy()
    for k,T in enumerate(tets):
        X = coords[T]
        detJ = np.linalg.det(np.vstack((X[1]-X[0], X[2]-X[0], X[3]-X[0])))
        if detJ < 0:
            tets[k,[1,2]] = tets[k,[2,1]]
    return tets


def alfeld_split_3d(coords, tets):
    """Alfeld split of every tetrahedron into 4 subtetrahedra.

    Adds one barycenter per original tet and connects it to each face.
    """
    nV = coords.shape[0]
    nT = tets.shape[0]
    bary = coords[tets].mean(axis=1)
    coordsA = np.vstack((coords, bary))

    subtets = []
    parentA = []
    for K,T in enumerate(tets):
        a,b,c,d = T
        m = nV + K
        # Four subtets, each opposite one original vertex.
        candidates = [
            [m,b,c,d],
            [a,m,c,d],
            [a,b,m,d],
            [a,b,c,m],
        ]
        for S in candidates:
            subtets.append(S)
            parentA.append(K)
    elemsA = orient_tets_positive(coordsA, np.array(subtets, dtype=int))
    parentA = np.array(parentA, dtype=int)

    # mu=0 at original vertices, mu=1 at barycenters.
    muA = np.concatenate((np.zeros(nV), np.ones(nT)))
    return coordsA, elemsA, parentA, muA


def barycentric_coords_tet(P, X):
    """Barycentric coordinates of points P with respect to tetrahedron X."""
    P = np.asarray(P, dtype=float)
    M = np.vstack((X.T, np.ones(4)))
    R = np.vstack((P.T, np.ones(P.shape[0])))
    lam = np.linalg.solve(M, R).T
    return lam


def local_z0_for_vertex_3d(coords, tets, coordsA, elemsA, parentA, muA, y_id):
    """Compute psi_0(y), piecewise-constant Z^0(y), and local checks."""
    nV = coords.shape[0]
    nTA = elemsA.shape[0]

    star_tets = np.flatnonzero(np.any(tets == y_id, axis=1))
    star_subtets = np.flatnonzero(np.isin(parentA, star_tets))

    local_nodes = np.unique(tets[star_tets].ravel())
    nloc = len(local_nodes)
    g2l = -np.ones(nV, dtype=int)
    g2l[local_nodes] = np.arange(nloc)

    A = np.zeros((nloc,nloc), dtype=float)
    m = np.zeros(nloc, dtype=float)
    star_vol = 0.0

    q = np.zeros(nloc, dtype=float)
    q[g2l[y_id]] = 1.0

    for K in star_tets:
        verts = tets[K]
        loc = g2l[verts]
        X = coords[verts]
        volK, grads = tet_volume_grads(X)
        star_vol += volK
        m[loc] += volK/4.0

        sub_ids = np.flatnonzero(parentA == K)
        int_mu_K = 0.0
        for s in sub_ids:
            S = elemsA[s]
            XA = coordsA[S]
            volS, _ = tet_volume_grads(XA)
            int_mu_K += volS * np.mean(muA[S])

        A[np.ix_(loc,loc)] += int_mu_K * (grads @ grads.T)

    b = q - m/star_vol

    KKT = np.block([[A, m[:,None]], [m[None,:], np.zeros((1,1))]])
    rhs = np.concatenate((b, [0.0]))
    sol = np.linalg.solve(KKT, rhs)
    c = sol[:nloc]
    lam = sol[-1]

    ell_weak = m/star_vol + A @ c

    z0_values = np.zeros(nTA, dtype=float)
    for K in star_tets:
        verts = tets[K]
        loc = g2l[verts]
        X = coords[verts]
        _, grads = tet_volume_grads(X)
        grad_psi = c[loc] @ grads

        for s in np.flatnonzero(parentA == K):
            S = elemsA[s]
            XA = coordsA[S]
            _, gradsA = tet_volume_grads(XA)
            grad_mu = muA[S] @ gradsA
            z0_values[s] = 1.0/star_vol - np.dot(grad_mu, grad_psi)

    ell_point = np.zeros(nloc, dtype=float)
    z0_integral = 0.0
    for K in star_tets:
        verts = tets[K]
        loc = g2l[verts]
        X = coords[verts]
        for s in np.flatnonzero(parentA == K):
            S = elemsA[s]
            XA = coordsA[S]
            volS, _ = tet_volume_grads(XA)
            lamS = barycentric_coords_tet(XA, X)
            int_parent_phi = volS * np.mean(lamS, axis=0)
            ell_point[loc] += z0_values[s] * int_parent_phi
            z0_integral += z0_values[s] * volS

    checks = {
        "sym_A": np.linalg.norm(A-A.T, ord="fro"),
        "null_A": np.linalg.norm(A @ np.ones(nloc), ord=np.inf),
        "rhs_compat": abs(np.sum(b)),
        "zero_mean_psi": abs(m @ c),
        "lambda": abs(lam),
        "repro_weak": np.linalg.norm(ell_weak-q, ord=np.inf),
        "repro_pointwise": np.linalg.norm(ell_point-q, ord=np.inf),
        "z0_integral": abs(z0_integral-1.0),
    }

    return {
        "star_tets": star_tets,
        "star_subtets": star_subtets,
        "local_nodes": local_nodes,
        "A": A,
        "m": m,
        "b": b,
        "q": q,
        "psi_coeffs": c,
        "lambda": lam,
        "ell_weak": ell_weak,
        "ell_point": ell_point,
        "z0_values": z0_values,
        "star_vol": star_vol,
        "checks": checks,
    }


def build_pi0_matrix_3d(coords, tets, coordsA, elemsA, parentA, muA):
    """Assemble P0: Alfeld P1 nodal values -> original P1 nodal values."""
    nV = coords.shape[0]
    nT = tets.shape[0]
    nVA = coordsA.shape[0]

    # E embeds original P1 nodal values into Alfeld P1 nodal values.
    rows = list(range(nV))
    cols = list(range(nV))
    vals = [1.0]*nV
    for K,T in enumerate(tets):
        bary_id = nV + K
        for v in T:
            rows.append(bary_id)
            cols.append(v)
            vals.append(0.25)
    E = coo_matrix((vals,(rows,cols)), shape=(nVA,nV)).tocsc()

    I = []
    J = []
    V = []
    max_checks = {
        "max_local_weak_repro": 0.0,
        "max_local_point_repro": 0.0,
        "max_z0_integral_error": 0.0,
        "max_zero_mean_error": 0.0,
        "max_null_error": 0.0,
    }

    local_data = []
    for y_id in range(nV):
        out = local_z0_for_vertex_3d(coords, tets, coordsA, elemsA, parentA, muA, y_id)
        local_data.append(out)
        ch = out["checks"]
        max_checks["max_local_weak_repro"] = max(max_checks["max_local_weak_repro"], ch["repro_weak"])
        max_checks["max_local_point_repro"] = max(max_checks["max_local_point_repro"], ch["repro_pointwise"])
        max_checks["max_z0_integral_error"] = max(max_checks["max_z0_integral_error"], ch["z0_integral"])
        max_checks["max_zero_mean_error"] = max(max_checks["max_zero_mean_error"], ch["zero_mean_psi"])
        max_checks["max_null_error"] = max(max_checks["max_null_error"], ch["null_A"])

        # P0(y,j) = int Z0(y) Phi_j^A dx.
        # On each Alfeld subtet, Z0 is constant and integral Phi_j^A = vol/4.
        for s in out["star_subtets"]:
            S = elemsA[s]
            volS, _ = tet_volume_grads(coordsA[S])
            val = out["z0_values"][s] * volS/4.0
            for a in S:
                I.append(y_id)
                J.append(a)
                V.append(val)

    P0 = coo_matrix((V,(I,J)), shape=(nV,nVA)).tocsc()
    return P0, E, max_checks, local_data


def unique_edges_from_tets(tets):
    edge_set = set()
    for T in tets:
        for i in range(4):
            for j in range(i+1,4):
                edge_set.add(tuple(sorted((int(T[i]), int(T[j])))))
    return sorted(edge_set)


def plot_tet_edges(ax, coords, tets, color="k", lw=0.6, alpha=0.8, max_edges=20000):
    edges = unique_edges_from_tets(tets)
    if len(edges) > max_edges:
        edges = edges[:max_edges]
    for i,j in edges:
        X = coords[[i,j]]
        ax.plot(X[:,0], X[:,1], X[:,2], color=color, lw=lw, alpha=alpha)
    ax.set_box_aspect((1,1,1))
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")


def make_plots(coords, tets, coordsA, elemsA, P0, E):
    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection="3d")
    plot_tet_edges(ax, coords, tets, color="k", lw=0.8)
    ax.set_title("Original tetrahedral mesh")
    plt.show()

    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection="3d")
    plot_tet_edges(ax, coordsA, elemsA, color="0.45", lw=0.45)
    ax.set_title("3D Alfeld split mesh")
    plt.show()

    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection="3d")
    plot_tet_edges(ax, coordsA, elemsA, color="0.75", lw=0.35)
    plot_tet_edges(ax, coords, tets, color="r", lw=1.0)
    ax.set_title("Alfeld split mesh with original mesh overlaid")
    plt.show()

    uA = (np.sin(2*np.pi*coordsA[:,0]) * np.cos(np.pi*coordsA[:,1])
          + coordsA[:,0]**2 + 0.25*coordsA[:,2])
    u0 = P0 @ uA
    uA_proj = E @ u0

    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(coordsA[:,0], coordsA[:,1], coordsA[:,2], c=uA, s=28)
    plot_tet_edges(ax, coordsA, elemsA, color="0.85", lw=0.25, alpha=0.5)
    ax.set_title("Sample input function on Alfeld nodes")
    plt.colorbar(sc, ax=ax, shrink=0.65)
    plt.show()

    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(coords[:,0], coords[:,1], coords[:,2], c=u0, s=45)
    plot_tet_edges(ax, coords, tets, color="0.7", lw=0.5, alpha=0.5)
    ax.set_title("Projected Pi_h^0 function on original nodes")
    plt.colorbar(sc, ax=ax, shrink=0.65)
    plt.show()

    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(coordsA[:,0], coordsA[:,1], coordsA[:,2], c=uA_proj, s=28)
    plot_tet_edges(ax, coordsA, elemsA, color="0.85", lw=0.25, alpha=0.5)
    ax.set_title("Projected function embedded back on Alfeld nodes")
    plt.colorbar(sc, ax=ax, shrink=0.65)
    plt.show()

    plt.figure(figsize=(7,5))
    plt.spy(P0, markersize=2)
    plt.title("Sparsity pattern of P0")
    plt.xlabel("Alfeld P1 input node")
    plt.ylabel("Original P1 output vertex")
    plt.show()


def run_demo(maxh=0.45, fallback_n=2, make_plots=True):
    coords, tets, ngmesh, source = make_mesh(maxh=maxh, fallback_n=fallback_n)
    coordsA, elemsA, parentA, muA = alfeld_split_3d(coords, tets)

    print("\n3D mesh source")
    print("  source                  :", source)
    print("  original vertices       :", coords.shape[0])
    print("  original tetrahedra     :", tets.shape[0])
    print("  original edges          :", len(unique_edges_from_tets(tets)))
    print("  Alfeld vertices         :", coordsA.shape[0])
    print("  Alfeld subtetrahedra    :", elemsA.shape[0])
    print("  Alfeld edges            :", len(unique_edges_from_tets(elemsA)))
    print("  split rule              : one barycenter per original tet, connected to its four faces")

    P0, E, data, local_data = build_pi0_matrix_3d(coords, tets, coordsA, elemsA, parentA, muA)

    PE = P0 @ E
    PEminusI = PE - eye(coords.shape[0], format="csc")
    err_PE_entry = np.max(np.abs(PEminusI.data)) if PEminusI.nnz else 0.0
    err_PE_inf = sparse_norm(PEminusI, ord=np.inf)

    Q = E @ P0
    err_idempotent = sparse_norm(Q @ Q - Q, ord=np.inf)

    err_const = np.max(np.abs(P0 @ np.ones(coordsA.shape[0]) - np.ones(coords.shape[0])))
    err_x = np.max(np.abs(P0 @ coordsA[:,0] - coords[:,0]))
    err_y = np.max(np.abs(P0 @ coordsA[:,1] - coords[:,1]))
    err_z = np.max(np.abs(P0 @ coordsA[:,2] - coords[:,2]))

    rng = np.random.default_rng(1)
    u0 = rng.standard_normal(coords.shape[0])
    err_random_p1 = np.max(np.abs(P0 @ (E @ u0) - u0))

    print("\nLocal Z0/psi checks over all vertices")
    print("  max weak reproduction          : %.3e" % data["max_local_weak_repro"])
    print("  max pointwise-Z0 reproduction  : %.3e" % data["max_local_point_repro"])
    print("  max |int_st Z0 - 1|            : %.3e" % data["max_z0_integral_error"])
    print("  max |int_st psi|               : %.3e" % data["max_zero_mean_error"])
    print("  max ||A*1||_inf                : %.3e" % data["max_null_error"])

    print("\nGlobal Pi0 matrix checks")
    print("  size(P0)                       : %d x %d" % P0.shape)
    print("  nnz(P0)                        : %d" % P0.nnz)
    print("  max entry |P0*E - I|           : %.3e" % err_PE_entry)
    print("  ||P0*E - I||_inf               : %.3e" % err_PE_inf)
    print("  ||(E*P0)^2 - E*P0||_inf        : %.3e" % err_idempotent)
    print("  constant reproduction          : %.3e" % err_const)
    print("  x-coordinate reproduction      : %.3e" % err_x)
    print("  y-coordinate reproduction      : %.3e" % err_y)
    print("  z-coordinate reproduction      : %.3e" % err_z)
    print("  random original-P1 reproduction: %.3e" % err_random_p1)

    if make_plots:
        make_plots_func = globals()["make_plots"]
        make_plots_func(coords, tets, coordsA, elemsA, P0, E)

    return {
        "coords": coords,
        "tets": tets,
        "coordsA": coordsA,
        "elemsA": elemsA,
        "parentA": parentA,
        "muA": muA,
        "P0": P0,
        "E": E,
        "data": data,
        "local_data": local_data,
        "ngsolve_mesh": ngmesh,
    }


if __name__ == "__main__":
    run_demo(maxh=0.45, fallback_n=2, make_plots=True)


In [ ]:
out = run_demo(maxh=0.45, fallback_n=2, make_plots=True)
